# ⚪ Notebook 6 — Evaluación Automática de Modelos con LLM-as-Judge
> **Cómo medir calidad cuando no hay ground truth fácil**

En este notebook vamos a:
1. Generar respuestas con un LLM (Gemini) para distintas preguntas
2. Evaluar la calidad con otro LLM actuando como juez
3. Comparar evaluación automática vs métricas clásicas (ROUGE-like)
4. Construir un sistema de evaluación continua para pipelines GenAI

**Módulo:** IA Generativa — Clase 4: Pipelines ML + GenAI

## 1. Instalación de dependencias

In [ ]:
!pip install langchain langchain-google-genai pandas numpy python-dotenv rouge-score -q

## 2. Configuración

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Modelo que genera respuestas
llm_generador = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.7  # más creatividad para generar variedad
)

# Modelo que evalúa (mismo modelo, rol diferente)
llm_juez = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0  # más determinista para evaluar
)

print("✅ Modelos configurados")
print("  Generador: gemini-1.5-flash (temperatura=0.7)")
print("  Juez:      gemini-1.5-flash (temperatura=0)")

## 3. Dataset de evaluación: preguntas con respuesta de referencia

In [ ]:
# Benchmark de preguntas con respuestas de referencia (ground truth)
benchmark = [
    {
        "id": 1,
        "pregunta": "¿Qué es el aprendizaje supervisado en Machine Learning?",
        "contexto": "Explicación para un estudiante universitario de primer año",
        "respuesta_referencia": (
            "El aprendizaje supervisado es un tipo de ML donde el modelo aprende "
            "a partir de datos etiquetados. Se le presentan ejemplos con inputs y "
            "sus correspondientes outputs correctos, y el modelo aprende a predecir "
            "el output para nuevos inputs. Ejemplos: clasificación de spam, predicción de precios."
        ),
        "criterios": ["precision_tecnica", "claridad", "ejemplos_concretos"]
    },
    {
        "id": 2,
        "pregunta": "Explica la diferencia entre overfitting y underfitting",
        "contexto": "Para alguien que está aprendiendo ML",
        "respuesta_referencia": (
            "Overfitting ocurre cuando el modelo aprende demasiado bien los datos de entrenamiento, "
            "incluyendo el ruido, y falla en datos nuevos. Underfitting ocurre cuando el modelo "
            "es demasiado simple y no logra capturar los patrones. El objetivo es encontrar "
            "el punto medio con buena generalización."
        ),
        "criterios": ["precision_tecnica", "claridad", "analogia_o_ejemplo"]
    },
    {
        "id": 3,
        "pregunta": "¿Cuándo usar un modelo de árbol de decisión vs una red neuronal?",
        "contexto": "Situación práctica de selección de modelo",
        "respuesta_referencia": (
            "Usar árbol de decisión cuando: datos tabulares pequeños/medianos, necesitas interpretabilidad, "
            "tienes pocas features, recursos computacionales limitados. "
            "Usar red neuronal cuando: datos masivos, imágenes/texto/audio, "
            "patrones muy complejos, tienes capacidad computacional."
        ),
        "criterios": ["precision_tecnica", "practicidad", "casos_de_uso"]
    },
    {
        "id": 4,
        "pregunta": "¿Qué es un pipeline en Machine Learning y para qué sirve?",
        "contexto": "Explicación técnica con ejemplos de código o flujo",
        "respuesta_referencia": (
            "Un pipeline ML es una secuencia de pasos de procesamiento de datos y modelado "
            "encadenados. Permite reproducibilidad, evita data leakage, y facilita "
            "despliegue en producción. En sklearn: Pipeline([('scaler', StandardScaler()), "
            "('clf', LogisticRegression())]). Se aplica fit/transform de forma consistente."
        ),
        "criterios": ["precision_tecnica", "ejemplo_practico", "ventajas"]
    },
]

print(f"Benchmark cargado: {len(benchmark)} preguntas")

## 4. Generar respuestas con distintos prompts (A/B test)

In [ ]:
# Variante A: prompt básico
prompt_basico = ChatPromptTemplate.from_template(
    "Responde esta pregunta: {pregunta}"
)

# Variante B: prompt optimizado con contexto
prompt_optimizado = ChatPromptTemplate.from_template("""
Eres un profesor experto en Machine Learning.
Contexto del estudiante: {contexto}

Pregunta: {pregunta}

Responde de forma clara, con al menos un ejemplo concreto,
y en no más de 4 oraciones.
""")

chain_a = prompt_basico | llm_generador | StrOutputParser()
chain_b = prompt_optimizado | llm_generador | StrOutputParser()

# Generar respuestas
print("Generando respuestas (Variante A y B)...")
respuestas = []

for item in benchmark:
    print(f"  Pregunta {item['id']}: {item['pregunta'][:50]}...")
    
    resp_a = chain_a.invoke({"pregunta": item["pregunta"]})
    resp_b = chain_b.invoke({"pregunta": item["pregunta"], "contexto": item["contexto"]})
    
    respuestas.append({
        "id": item["id"],
        "pregunta": item["pregunta"],
        "respuesta_referencia": item["respuesta_referencia"],
        "criterios": item["criterios"],
        "respuesta_a": resp_a,
        "respuesta_b": resp_b,
    })
    print("    ✅")

print(f"\n✅ {len(respuestas) * 2} respuestas generadas")

## 5. LLM-as-Judge: Evaluación automática de calidad

In [ ]:
prompt_juez = ChatPromptTemplate.from_template("""
Eres un evaluador experto en pedagogía y Machine Learning.
Evalúa la calidad de esta respuesta según los criterios dados.

**Pregunta:** {pregunta}
**Respuesta de referencia:** {referencia}
**Respuesta a evaluar:** {respuesta}
**Criterios de evaluación:** {criterios}

Evalúa y responde ÚNICAMENTE con este JSON:
{{
  "puntuacion_global": número entre 1 y 10,
  "precision_tecnica": número entre 1 y 10,
  "claridad": número entre 1 y 10,
  "completitud": número entre 1 y 10,
  "fortalezas": "lista de 2 fortalezas separadas por coma",
  "debilidades": "lista de 2 debilidades o N/A separadas por coma",
  "justificacion": "máximo 2 oraciones"
}}
""")

chain_juez = prompt_juez | llm_juez | StrOutputParser()

def evaluar_respuesta(pregunta, referencia, respuesta, criterios):
    raw = chain_juez.invoke({
        "pregunta": pregunta,
        "referencia": referencia,
        "respuesta": respuesta,
        "criterios": ", ".join(criterios)
    })
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

# Evaluar todas las respuestas
print("Evaluando respuestas con LLM-as-Judge...")
evaluaciones = []

for item in respuestas:
    print(f"  Pregunta {item['id']}:", end=" ")
    
    eval_a = evaluar_respuesta(item["pregunta"], item["respuesta_referencia"],
                                item["respuesta_a"], item["criterios"])
    eval_b = evaluar_respuesta(item["pregunta"], item["respuesta_referencia"],
                                item["respuesta_b"], item["criterios"])
    
    evaluaciones.append({
        "id": item["id"],
        "variante_a": eval_a,
        "variante_b": eval_b,
        "respuesta_a": item["respuesta_a"],
        "respuesta_b": item["respuesta_b"],
        "pregunta": item["pregunta"]
    })
    print(f"A={eval_a['puntuacion_global']}/10, B={eval_b['puntuacion_global']}/10 ✅")

print("\n✅ Evaluaciones completadas")

## 6. Métricas clásicas: similitud de texto (ROUGE simplificado)

In [ ]:
def rouge1_simple(hipotesis: str, referencia: str) -> float:
    """Implementación simplificada de ROUGE-1 (overlap de unigramas)."""
    tokens_hip = set(hipotesis.lower().split())
    tokens_ref = set(referencia.lower().split())
    overlap = len(tokens_hip & tokens_ref)
    if len(tokens_ref) == 0:
        return 0.0
    return overlap / len(tokens_ref)

# Comparar métricas clásicas vs LLM-as-Judge
print("=" * 65)
print(f"{'ID':<4} {'ROUGE-A':>8} {'ROUGE-B':>8} {'LLM-A':>7} {'LLM-B':>7} {'Ganador'}")
print("-" * 65)

for i, (eval_item, resp_item) in enumerate(zip(evaluaciones, respuestas)):
    rouge_a = rouge1_simple(resp_item["respuesta_a"], resp_item["respuesta_referencia"])
    rouge_b = rouge1_simple(resp_item["respuesta_b"], resp_item["respuesta_referencia"])
    llm_a = eval_item["variante_a"]["puntuacion_global"]
    llm_b = eval_item["variante_b"]["puntuacion_global"]
    
    rouge_ganador = "A" if rouge_a > rouge_b else "B"
    llm_ganador = "A" if llm_a > llm_b else "B"
    emoji = "✅" if rouge_ganador == llm_ganador else "⚠️ difieren"
    
    print(f"{eval_item['id']:<4} {rouge_a:>8.3f} {rouge_b:>8.3f} {llm_a:>7.1f} {llm_b:>7.1f}   ROUGE:{rouge_ganador} / LLM:{llm_ganador} {emoji}")

print("-" * 65)
print("\n⚠️ Cuando ROUGE y LLM-Judge difieren, el LLM suele ser más preciso")
print("   ROUGE mide similitud léxica, no calidad semántica real")

## 7. Reporte final comparativo A vs B

In [ ]:
scores_a = [e["variante_a"]["puntuacion_global"] for e in evaluaciones]
scores_b = [e["variante_b"]["puntuacion_global"] for e in evaluaciones]

print("\n" + "="*55)
print("📊 REPORTE FINAL: PROMPT A vs PROMPT B")
print("="*55)
print(f"\nPrompt A (básico):    media = {np.mean(scores_a):.2f}/10")
print(f"Prompt B (optimizado): media = {np.mean(scores_b):.2f}/10")
print(f"Mejora: {np.mean(scores_b) - np.mean(scores_a):+.2f} puntos")

print("\n📋 DETALLE POR DIMENSIÓN (Prompt B):")
dims = ["precision_tecnica", "claridad", "completitud"]
for dim in dims:
    scores = [e["variante_b"][dim] for e in evaluaciones]
    print(f"  {dim:<22} {np.mean(scores):.1f}/10")

print("\n🔍 ANÁLISIS POR PREGUNTA:")
for e in evaluaciones:
    diff = e['variante_b']['puntuacion_global'] - e['variante_a']['puntuacion_global']
    emoji = "📈" if diff > 0 else ("📉" if diff < 0 else "➡️")
    print(f"  P{e['id']}: {emoji} B mejora en {diff:+.0f} pts")
    print(f"     Fortaleza B: {e['variante_b']['fortalezas']}")

## 8. Sistema de evaluación continua (función reutilizable)

In [ ]:
def evaluar_pipeline_llm(chain, benchmark_items: list, nombre: str = "Pipeline") -> dict:
    """
    Función genérica para evaluar cualquier chain de LangChain
    contra un benchmark con LLM-as-Judge.
    
    Úsala para evaluar tus propios pipelines!
    """
    resultados = []
    
    for item in benchmark_items:
        respuesta = chain.invoke({"pregunta": item["pregunta"]})
        evaluacion = evaluar_respuesta(
            item["pregunta"],
            item["respuesta_referencia"],
            respuesta,
            item["criterios"]
        )
        resultados.append(evaluacion)
    
    scores = [r["puntuacion_global"] for r in resultados]
    
    return {
        "nombre": nombre,
        "score_promedio": round(np.mean(scores), 2),
        "score_min": min(scores),
        "score_max": max(scores),
        "evaluaciones": resultados
    }

# Ejemplo de uso
print("✅ Función evaluar_pipeline_llm() lista para reutilizar")
print("   Úsala para comparar cualquier par de prompts o modelos")
print("\nEjemplo:")
print("   resultado = evaluar_pipeline_llm(mi_chain, benchmark, 'Mi Pipeline v2')")
print("   print(resultado['score_promedio'])")

## 🎯 Conclusiones

**LLM-as-Judge** es el estándar actual para evaluar sistemas GenAI:

| Método | Qué mide | Limitaciones |
|--------|----------|-------------|
| ROUGE | Overlap de palabras | No mide calidad semántica real |
| BLEU | Similitud n-gramas | Diseñado para traducción, no QA |
| LLM-Judge | Calidad, razonamiento, utilidad | Costo de API, sesgo del modelo |

**Patrón clave:** `Generar respuestas → LLM-Judge evalúa → Métricas + justificación → Decisión A/B`

Esto permite un **ciclo de mejora continua** de tus prompts y pipelines sin necesidad de anotación humana costosa.